# Colab 实验启动器（GitHub + Google Drive 自动回传）

使用步骤：
1. 在菜单栏点击 **代码执行程序 → 更改运行时类型 → T4 GPU**。
2. 在下方第一个代码单元格填好你的 `GITHUB_REPO`（格式：`用户名/仓库名`）。
3. 如需访问私有仓库或自动将结果推回 GitHub，请在左侧 **钥匙图标 (Secrets)** 添加 `GITHUB_TOKEN`。
4. 点击菜单栏 **代码执行程序 → 全部运行**。

In [ ]:
#@title 1. 实验与同步参数配置
GITHUB_REPO = "luyiming123/Deep_Learning_Study"  #@param {type:"string"}
GITHUB_BRANCH = "main"  #@param {type:"string"}
EXPERIMENT_SCRIPT = "src/experiments/mlp_concise.py"  #@param {type:"string"}
CONFIG_PATH = "configs/mlp_concise.yaml"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/CoLab_Outputs"  #@param {type:"string"}
PUSH_RESULTS_TO_GITHUB = False  #@param {type:"boolean"}
GIT_USER_NAME = "Colab Runner"  #@param {type:"string"}
GIT_USER_EMAIL = "colab-runner@users.noreply.github.com"  #@param {type:"string"}

In [ ]:
#@title 2. 检查 GPU 并挂载 Google Drive
import subprocess
import torch

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
#@title 3. 从 GitHub 拉取最新代码与配置
import os
import subprocess
from pathlib import Path

if "your-username" in GITHUB_REPO:
    raise ValueError("请先在单元格 1 中将 GITHUB_REPO 替换为你的真实 GitHub 仓库（用户名/仓库名）")

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None

if GITHUB_TOKEN:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
else:
    repo_url = f"https://github.com/{GITHUB_REPO}.git"

REPO_DIR = Path("/content/CoLab")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", repo_url], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--rebase", "origin", GITHUB_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", GITHUB_BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("当前工作目录:", Path.cwd())

In [ ]:
#@title 4. 运行实验
from datetime import datetime
from pathlib import Path
import subprocess
import sys
import yaml

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyYAML"], check=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    exp_cfg = yaml.safe_load(f)

exp_name = exp_cfg["experiment_name"]
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

if SAVE_TO_DRIVE:
    RUN_OUTPUT_DIR = Path(DRIVE_OUTPUT_ROOT) / exp_name / run_id
else:
    RUN_OUTPUT_DIR = Path("/content/CoLab/results") / exp_name / run_id

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("本次完整输出目录:", RUN_OUTPUT_DIR)

cmd = [
    sys.executable,
    EXPERIMENT_SCRIPT,
    "--config", CONFIG_PATH,
    "--data-dir", "/content/data",
    "--output-dir", str(RUN_OUTPUT_DIR),
]
subprocess.run(cmd, check=True)

In [ ]:
#@title 5. 查看训练曲线与实验总结
from IPython.display import Image, Markdown, display

summary_file = RUN_OUTPUT_DIR / "summary.md"
curve_file = RUN_OUTPUT_DIR / "learning_curve.png"

if summary_file.exists():
    display(Markdown(summary_file.read_text(encoding="utf-8")))
if curve_file.exists():
    display(Image(filename=str(curve_file)))

In [ ]:
#@title 6. 同步轻量结果回 GitHub 仓库（可选）
import shutil
import subprocess
from pathlib import Path

LIGHTWEIGHT_FILES = ["metrics.json", "summary.md", "learning_curve.png"]
repo_result_dir = Path("/content/CoLab/results") / exp_name / run_id
repo_result_dir.mkdir(parents=True, exist_ok=True)

for fname in LIGHTWEIGHT_FILES:
    src_file = RUN_OUTPUT_DIR / fname
    if src_file.exists() and src_file.resolve() != (repo_result_dir / fname).resolve():
        shutil.copy2(src_file, repo_result_dir / fname)

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        raise RuntimeError("已开启 PUSH_RESULTS_TO_GITHUB，但未在 Colab Secrets 中找到 GITHUB_TOKEN。")
    subprocess.run(["git", "config", "user.name", GIT_USER_NAME], check=True)
    subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL], check=True)
    subprocess.run(["git", "add", str(repo_result_dir)], check=True)
    subprocess.run(["git", "commit", "-m", f"results({exp_name}): colab run {run_id}"], check=True)
    subprocess.run(["git", "push", "origin", GITHUB_BRANCH], check=True)
    print("已将轻量结果推送回 GitHub，本地执行 .\\scripts\\sync.ps1 即可拉取！")
else:
    print("未开启自动推送 GitHub。所有产物已保存在:", RUN_OUTPUT_DIR)